In [7]:
import os
import streamlit as st 
import pickle
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()
llm=ChatGroq(
    model="openai/gpt-oss-120b", 
    temperature=0.7, 
    max_tokens=500
)


In [23]:
prompt = ChatPromptTemplate.from_template("""
Bạn là trợ lý trả lời câu hỏi dựa trên tài liệu.

Chỉ dùng thông tin trong Context để trả lời.
Nếu không có thông tin, hãy nói: "Tôi không tìm thấy thông tin trong tài liệu."

Context:
{context}

Câu hỏi:
{question}
""")

In [26]:
chain= prompt|llm|StrOutputParser()

### Load data

In [10]:
urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
]
loader=UnstructuredURLLoader(urls=urls)
data=loader.load()

In [11]:
len(data)

2

### Split data to create chunks

In [12]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)
docs=text_splitter.split_documents(data)

In [16]:
len(docs)

19

In [17]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nMC Learn\n\nGold Loan\n\nGold Loan\n\nTrending Topics\n

### Create embeddings for these chunks and save them to FAISS index

In [15]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
vectorstore=FAISS.from_documents(docs,embeddings)


In [20]:
file_path="vector_index.pkl"
with open(file_path,"wb") as f:
    pickle.dump(vectorstore,f)

In [21]:
if os.path.exists(file_path):
    with open(file_path,"rb") as f:
        vectorIndex=pickle.load(f)

In [33]:
retriever=vectorIndex.as_retriever(search_kwargs={"k":3})
question= "What is the price of Tiago iCNG ?"
retrieved_docs=retriever.invoke(question)
context= "\n\n".join(doc.page_content for doc in retrieved_docs)
answer=chain.invoke({
    "context":context,
    "question":question
})
for doc in retrieved_docs:
    print(doc.metadata.get("source"))
print(answer)

https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
The Tiago iCNG is priced between **Rs 6.55 lakh and Rs 8.1 lakh**.
